# 0. MACCS Keys fingerprints and the PLS model

Trains the partial least squares (PLS) regression model on the ten solvents with
measured yields and predicts the yield for all 29 candidate solvents.

| Output | Corresponding item in the paper |
|---|---|
| `output/train_MACCSKeys.csv` | fingerprints of the training solvents (SI) |
| `output/pls_model.joblib` | the trained model, read by notebooks 3, 4 and 5 |
| `output/pred_yield.csv` | Table S4 (predicted yields for all 29 solvents) |

**Input:** `data/solvent.csv` with columns `name`, `smiles`, `exp_yield`.
Rows with an empty `exp_yield` are candidate solvents that have not been measured;
they are excluded from training but retained as prediction targets.

In [ ]:
import joblib
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem.MACCSkeys import GenMACCSKeys
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler

from pathlib import Path

# Resolve paths relative to the repository root, so that the notebook runs
# both from notebooks/ and from the repository root.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
QM = ROOT / "qm" / "qm_nbo_t6311++g"
OUT = ROOT / "output"
OUT.mkdir(exist_ok=True)

N_COMPONENTS = 3      # number of PLS latent variables used throughout the paper


def maccs_matrix(smiles_list):
    """Convert a list of SMILES into a (n_molecules, 167) MACCS Keys matrix.

    Bit 0 is a padding bit that RDKit always leaves at zero, so only bits
    1-166 carry structural information.
    """
    mols = [Chem.MolFromSmiles(s) for s in smiles_list]
    if any(m is None for m in mols):
        bad = [s for s, m in zip(smiles_list, mols) if m is None]
        raise ValueError(f"RDKit could not parse: {bad}")
    return np.array([GenMACCSKeys(m) for m in mols], dtype=float)


print("root:", ROOT)

## Load the solvent list

`solvent` holds all 29 candidates; `train` holds the ten with an experimental yield.
Both are kept, because the model is trained on `train` and applied to `solvent`.

In [ ]:
solvent = pd.read_csv(DATA / "solvent.csv")
train = solvent.dropna(subset=["exp_yield"]).reset_index(drop=True)

print(f"{len(solvent)} candidate solvents, {len(train)} with a measured yield")
train

## Compute the MACCS Keys fingerprints

Each solvent becomes a 167-bit binary vector indicating the presence or absence of
predefined substructures. The fingerprints of the training solvents are written out
as a machine-readable file.

In [ ]:
x_train = pd.DataFrame(maccs_matrix(train["smiles"]).astype(int),
                       index=train["name"])
x_train.to_csv(OUT / "train_MACCSKeys.csv")

y_train = train[["exp_yield"]].set_index(train["name"])

display(x_train.head())
display(y_train.head())

## Train the PLS model

Both the descriptors and the response are standardized before fitting. Standardizing
the binary fingerprint removes the bits that are constant across the training set
(they have zero variance and are mapped to zero), so only the bits that actually vary
among the ten solvents can contribute to the regression.

The model is stored with `joblib`, the standard format for scikit-learn estimators.
Saving an estimator with `numpy.save` requires `allow_pickle=True` on reload and
breaks across NumPy versions.

In [ ]:
x_scaler, y_scaler = StandardScaler(), StandardScaler()
x_scaled = x_scaler.fit_transform(x_train)
y_scaled = y_scaler.fit_transform(y_train)

model = PLSRegression(N_COMPONENTS).fit(x_scaled, y_scaled)

# the scalers are stored together with the model, because the predictions
# cannot be reproduced without them
joblib.dump({"model": model, "x_scaler": x_scaler, "y_scaler": y_scaler,
             "train_names": list(train["name"]), "n_components": N_COMPONENTS},
            OUT / "pls_model.joblib")

print(f"PLS model with {N_COMPONENTS} latent variables saved to "
      f"{(OUT / 'pls_model.joblib').relative_to(ROOT)}")

## Predict the yield for all 29 candidate solvents

The ten training solvents are included in the prediction set; their predicted values
indicate how closely the model reproduces the data it was fitted to. Predictions are
not constrained to the 0-100% range, so values slightly outside it can occur and
should be read as "at the top of the ranking" rather than as literal yields.

In [ ]:
pred_scaled = model.predict(x_scaler.transform(maccs_matrix(solvent["smiles"])))

y_pred = pd.DataFrame(y_scaler.inverse_transform(pred_scaled),
                      columns=["pred_yield"], index=solvent["name"])
y_pred = y_pred.sort_values("pred_yield", ascending=False)
y_pred.to_csv(OUT / "pred_yield.csv")
y_pred

Note that several pairs of solvents receive exactly the same predicted yield.
This is not a numerical coincidence: the MACCS bits that distinguish them do not vary
across the ten training solvents, so the model cannot use them. See
`5_degeneracy_analysis.ipynb`.